In [1]:
# ============================================================
# PHASE 3.0 — BUILD SHARED MULTIMODAL DATASET
# Protein ProtBERT sliding-window + Genomic K3/K4/Basic
# ============================================================

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 150)

In [2]:
# ============================================================
# PATHS
# ============================================================
import os
from pathlib import Path
import pandas as pd
from google.colab import drive

try:
    drive.mount('/content/drive')
except ValueError:
    print("Drive đã được kết nối từ trước.")

PROJECT_DIR = Path("/content/drive/MyDrive/Project_Protein")

# Protein branch: ProtBERT sliding-window
PROTBERT_SW_DIR = PROJECT_DIR / "model" / "phase1_2_protbert_sliding_window_embedding"
PROTBERT_SW_EMBED_DIR = PROTBERT_SW_DIR / "embeddings"

# Genomic branch: K3 + K4 + Basic
PHASE2_DIR = PROJECT_DIR / "model" / "phase2_genomic_regulatory_baseline"
GENOMIC_FEATURE_DIR = PHASE2_DIR / "features"

# New Phase 3 directory
PHASE3_DIR = PROJECT_DIR / "model" / "phase3_multimodal_integration"
PHASE3_DATA_DIR = PHASE3_DIR / "shared_dataset"
PHASE3_FEATURE_DIR = PHASE3_DIR / "features"
PHASE3_RESULT_DIR = PHASE3_DIR / "results"
PHASE3_REPORT_DIR = PHASE3_DIR / "reports"

for folder in [
    PHASE3_DIR,
    PHASE3_DATA_DIR,
    PHASE3_FEATURE_DIR,
    PHASE3_RESULT_DIR,
    PHASE3_REPORT_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Phase 3 directory:", PHASE3_DIR)

Mounted at /content/drive
Phase 3 directory: /content/drive/MyDrive/Project_Protein/model/phase3_multimodal_integration


In [3]:
# ============================================================
# LOAD PROTBERT SLIDING-WINDOW EMBEDDINGS + METADATA
# ============================================================

protein_files = {
    "train": {
        "embedding": PROTBERT_SW_EMBED_DIR / "protbert_sw_train_embeddings.npy",
        "metadata": PROTBERT_SW_EMBED_DIR / "protbert_sw_train_metadata.csv"
    },
    "validation": {
        "embedding": PROTBERT_SW_EMBED_DIR / "protbert_sw_val_embeddings.npy",
        "metadata": PROTBERT_SW_EMBED_DIR / "protbert_sw_val_metadata.csv"
    },
    "test": {
        "embedding": PROTBERT_SW_EMBED_DIR / "protbert_sw_test_embeddings.npy",
        "metadata": PROTBERT_SW_EMBED_DIR / "protbert_sw_test_metadata.csv"
    }
}

for split_name, paths in protein_files.items():
    print(split_name)
    print("embedding exists:", paths["embedding"].exists(), paths["embedding"])
    print("metadata exists:", paths["metadata"].exists(), paths["metadata"])

train
embedding exists: True /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/embeddings/protbert_sw_train_embeddings.npy
metadata exists: True /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/embeddings/protbert_sw_train_metadata.csv
validation
embedding exists: True /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/embeddings/protbert_sw_val_embeddings.npy
metadata exists: True /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/embeddings/protbert_sw_val_metadata.csv
test
embedding exists: True /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/embeddings/protbert_sw_test_embeddings.npy
metadata exists: True /content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/embeddings/protbert_sw_test_metadata.csv


In [4]:
def load_protein_split(split_name, embedding_path, metadata_path):
    X = np.load(embedding_path)
    meta = pd.read_csv(metadata_path)

    assert X.shape[0] == meta.shape[0], f"Mismatch in {split_name}: X rows != metadata rows"

    meta = meta.copy()
    meta["original_protein_split"] = split_name

    return X, meta


protein_X_list = []
protein_meta_list = []

for split_name, paths in protein_files.items():
    X_split, meta_split = load_protein_split(
        split_name=split_name,
        embedding_path=paths["embedding"],
        metadata_path=paths["metadata"]
    )

    protein_X_list.append(X_split)
    protein_meta_list.append(meta_split)

protein_X_all = np.vstack(protein_X_list)
protein_meta_all = pd.concat(protein_meta_list, ignore_index=True)

print("Protein embedding matrix:", protein_X_all.shape)
print("Protein metadata:", protein_meta_all.shape)
print("Protein columns:")
print(protein_meta_all.columns.tolist())

display(protein_meta_all.head())

Protein embedding matrix: (1820, 1024)
Protein metadata: (1820, 11)
Protein columns:
['row_index', 'gene_id', 'gene_symbol', 'uniprot_accession', 'label', 'sequence_clean_length', 'n_windows', 'window_size', 'stride', 'representation', 'original_protein_split']


,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,n_windows,window_size,stride,representation,original_protein_split
0,0,ENSG00000205155,PSENEN,Q9NZ42,0,101,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022,train
1,1,ENSG00000164530,PI16,Q6UXB8,1,463,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022,train
2,2,ENSG00000143167,GPA33,Q99795,0,319,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022,train
3,3,ENSG00000137691,CFAP300,Q9BRQ4,0,267,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022,train
4,4,ENSG00000095981,KCNK16,Q96T55,1,309,1,1022,1022,ProtBERT_sliding_window_mean_pool_1022_stride_1022,train


In [5]:
# ============================================================
# STANDARDIZE PROTEIN METADATA
# ============================================================

def find_col(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


protein_gene_id_col = find_col(protein_meta_all, ["gene_id", "ensembl_gene_id"])
protein_gene_symbol_col = find_col(protein_meta_all, ["gene_symbol", "ensembl_gene_name"])
protein_label_col = find_col(protein_meta_all, ["label", "true_label"])

assert protein_gene_id_col is not None, "Protein gene_id column not found."
assert protein_gene_symbol_col is not None, "Protein gene_symbol column not found."
assert protein_label_col is not None, "Protein label column not found."

protein_meta_std = protein_meta_all.copy()

protein_meta_std = protein_meta_std.rename(columns={
    protein_gene_id_col: "gene_id",
    protein_gene_symbol_col: "gene_symbol",
    protein_label_col: "label"
})

protein_meta_std["label"] = protein_meta_std["label"].astype(int)

# Add row index so we can retrieve embeddings after merge
protein_meta_std["protein_row_index"] = np.arange(len(protein_meta_std))

print("Protein unique genes:", protein_meta_std["gene_id"].nunique())
print("Protein label counts:")
print(protein_meta_std["label"].value_counts().sort_index())

# Duplicate check
print("Duplicate protein gene_id rows:", protein_meta_std["gene_id"].duplicated().sum())

if protein_meta_std["gene_id"].duplicated().sum() > 0:
    display(
        protein_meta_std[
            protein_meta_std["gene_id"].duplicated(keep=False)
        ][["gene_id", "gene_symbol", "label", "original_protein_split"]].sort_values("gene_id")
    )

Protein unique genes: 1820
Protein label counts:
label
0    910
1    910
Name: count, dtype: int64
Duplicate protein gene_id rows: 0


In [6]:
# ============================================================
# LOAD GENOMIC FEATURES + METADATA
# ============================================================

genomic_files = {
    "train": {
        "features": GENOMIC_FEATURE_DIR / "X_train_k3_k4_plus_basic_v1.csv",
        "metadata": GENOMIC_FEATURE_DIR / "train_genomic_metadata_v1.csv"
    },
    "validation": {
        "features": GENOMIC_FEATURE_DIR / "X_val_k3_k4_plus_basic_v1.csv",
        "metadata": GENOMIC_FEATURE_DIR / "val_genomic_metadata_v1.csv"
    },
    "test": {
        "features": GENOMIC_FEATURE_DIR / "X_test_k3_k4_plus_basic_v1.csv",
        "metadata": GENOMIC_FEATURE_DIR / "test_genomic_metadata_v1.csv"
    }
}

for split_name, paths in genomic_files.items():
    print(split_name)
    print("features exists:", paths["features"].exists(), paths["features"])
    print("metadata exists:", paths["metadata"].exists(), paths["metadata"])

train
features exists: True /content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/features/X_train_k3_k4_plus_basic_v1.csv
metadata exists: True /content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/features/train_genomic_metadata_v1.csv
validation
features exists: True /content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/features/X_val_k3_k4_plus_basic_v1.csv
metadata exists: True /content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/features/val_genomic_metadata_v1.csv
test
features exists: True /content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/features/X_test_k3_k4_plus_basic_v1.csv
metadata exists: True /content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/features/test_genomic_metadata_v1.csv


In [7]:
def load_genomic_split(split_name, feature_path, metadata_path):
    X = pd.read_csv(feature_path)
    meta = pd.read_csv(metadata_path)

    assert X.shape[0] == meta.shape[0], f"Mismatch in {split_name}: X rows != metadata rows"

    meta = meta.copy()
    meta["original_genomic_split"] = split_name

    return X, meta


genomic_X_list = []
genomic_meta_list = []

for split_name, paths in genomic_files.items():
    X_split, meta_split = load_genomic_split(
        split_name=split_name,
        feature_path=paths["features"],
        metadata_path=paths["metadata"]
    )

    genomic_X_list.append(X_split)
    genomic_meta_list.append(meta_split)

genomic_X_all_df = pd.concat(genomic_X_list, ignore_index=True)
genomic_meta_all = pd.concat(genomic_meta_list, ignore_index=True)

print("Genomic feature matrix:", genomic_X_all_df.shape)
print("Genomic metadata:", genomic_meta_all.shape)
print("Genomic columns:")
print(genomic_meta_all.columns.tolist())

display(genomic_meta_all.head())

Genomic feature matrix: (1806, 356)
Genomic metadata: (1806, 4)
Genomic columns:
['gene_id', 'gene_symbol', 'label', 'original_genomic_split']


,gene_id,gene_symbol,label,original_genomic_split
0,ENSG00000149201,CCDC81,0,train
1,ENSG00000127324,TSPAN8,1,train
2,ENSG00000170262,MRAP,0,train
3,ENSG00000173950,XXYLT1,0,train
4,ENSG00000167792,NDUFV1,1,train


In [8]:
# ============================================================
# STANDARDIZE GENOMIC METADATA
# ============================================================

genomic_gene_id_col = find_col(genomic_meta_all, ["gene_id", "ensembl_gene_id"])
genomic_gene_symbol_col = find_col(genomic_meta_all, ["gene_symbol", "ensembl_gene_name"])
genomic_label_col = find_col(genomic_meta_all, ["label", "true_label"])

assert genomic_gene_id_col is not None, "Genomic gene_id column not found."
assert genomic_gene_symbol_col is not None, "Genomic gene_symbol column not found."
assert genomic_label_col is not None, "Genomic label column not found."

genomic_meta_std = genomic_meta_all.copy()

genomic_meta_std = genomic_meta_std.rename(columns={
    genomic_gene_id_col: "gene_id",
    genomic_gene_symbol_col: "gene_symbol",
    genomic_label_col: "label"
})

genomic_meta_std["label"] = genomic_meta_std["label"].astype(int)

# Add row index so we can retrieve features after merge
genomic_meta_std["genomic_row_index"] = np.arange(len(genomic_meta_std))

print("Genomic unique genes:", genomic_meta_std["gene_id"].nunique())
print("Genomic label counts:")
print(genomic_meta_std["label"].value_counts().sort_index())

print("Duplicate genomic gene_id rows:", genomic_meta_std["gene_id"].duplicated().sum())

if genomic_meta_std["gene_id"].duplicated().sum() > 0:
    display(
        genomic_meta_std[
            genomic_meta_std["gene_id"].duplicated(keep=False)
        ][["gene_id", "gene_symbol", "label", "original_genomic_split"]].sort_values("gene_id")
    )

Genomic unique genes: 1806
Genomic label counts:
label
0    903
1    903
Name: count, dtype: int64
Duplicate genomic gene_id rows: 0


In [9]:
# ============================================================
# MERGE SHARED MULTIMODAL GENE SET BY gene_id
# ============================================================

shared_meta = protein_meta_std[[
    "gene_id",
    "gene_symbol",
    "label",
    "protein_row_index",
    "original_protein_split"
]].merge(
    genomic_meta_std[[
        "gene_id",
        "gene_symbol",
        "label",
        "genomic_row_index",
        "original_genomic_split"
    ]],
    on="gene_id",
    how="inner",
    suffixes=("_protein", "_genomic")
)

print("Protein genes:", protein_meta_std["gene_id"].nunique())
print("Genomic genes:", genomic_meta_std["gene_id"].nunique())
print("Shared genes:", shared_meta.shape[0])

display(shared_meta.head())

Protein genes: 1820
Genomic genes: 1806
Shared genes: 1806


,gene_id,gene_symbol_protein,label_protein,protein_row_index,original_protein_split,gene_symbol_genomic,label_genomic,genomic_row_index,original_genomic_split
0,ENSG00000205155,PSENEN,0,0,train,PSENEN,0,1589,test
1,ENSG00000164530,PI16,1,1,train,PI16,1,250,train
2,ENSG00000143167,GPA33,0,2,train,GPA33,0,1634,test
3,ENSG00000137691,CFAP300,0,3,train,CFAP300,0,511,train
4,ENSG00000095981,KCNK16,1,4,train,KCNK16,1,837,train


In [10]:
# ============================================================
# LABEL CONSISTENCY CHECK
# ============================================================

shared_meta["label_match"] = (
    shared_meta["label_protein"] == shared_meta["label_genomic"]
)

print("Label match counts:")
print(shared_meta["label_match"].value_counts())

if not shared_meta["label_match"].all():
    mismatch_df = shared_meta[~shared_meta["label_match"]].copy()
    display(mismatch_df[[
        "gene_id",
        "gene_symbol_protein",
        "gene_symbol_genomic",
        "label_protein",
        "label_genomic"
    ]])

    # For safety: remove mismatched labels
    shared_meta = shared_meta[shared_meta["label_match"]].copy()

shared_meta["label"] = shared_meta["label_protein"].astype(int)

# Final gene symbol
shared_meta["gene_symbol"] = shared_meta["gene_symbol_genomic"].fillna(
    shared_meta["gene_symbol_protein"]
)

print("Final shared genes:", shared_meta.shape[0])
print("Shared label counts:")
print(shared_meta["label"].value_counts().sort_index())

Label match counts:
label_match
True    1806
Name: count, dtype: int64
Final shared genes: 1806
Shared label counts:
label
0    903
1    903
Name: count, dtype: int64


In [11]:
# ============================================================
# BUILD SHARED PROTEIN + GENOMIC FEATURE MATRICES
# ============================================================

protein_indices = shared_meta["protein_row_index"].astype(int).values
genomic_indices = shared_meta["genomic_row_index"].astype(int).values

X_protein_shared = protein_X_all[protein_indices]
X_genomic_shared = genomic_X_all_df.iloc[genomic_indices].reset_index(drop=True)

y_shared = shared_meta["label"].astype(int).values

print("X_protein_shared:", X_protein_shared.shape)
print("X_genomic_shared:", X_genomic_shared.shape)
print("y_shared:", y_shared.shape)

assert X_protein_shared.shape[0] == X_genomic_shared.shape[0] == len(y_shared)
assert X_genomic_shared.isna().sum().sum() == 0
assert np.isinf(X_genomic_shared.values).sum() == 0

# Combined raw concatenation
X_combined_shared = np.hstack([
    X_protein_shared,
    X_genomic_shared.values
])

print("X_combined_shared:", X_combined_shared.shape)

X_protein_shared: (1806, 1024)
X_genomic_shared: (1806, 356)
y_shared: (1806,)
X_combined_shared: (1806, 1380)


In [12]:
# ============================================================
# SHARED MULTIMODAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

RANDOM_STATE = 42

shared_indices = np.arange(len(shared_meta))

train_idx, test_val_idx = train_test_split(
    shared_indices,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_shared
)

val_idx, test_idx = train_test_split(
    test_val_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_shared[test_val_idx]
)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

print("\nTrain labels:")
print(pd.Series(y_shared[train_idx]).value_counts().sort_index())

print("\nValidation labels:")
print(pd.Series(y_shared[val_idx]).value_counts().sort_index())

print("\nTest labels:")
print(pd.Series(y_shared[test_idx]).value_counts().sort_index())

Train: 1264
Validation: 271
Test: 271

Train labels:
0    632
1    632
Name: count, dtype: int64

Validation labels:
0    135
1    136
Name: count, dtype: int64

Test labels:
0    136
1    135
Name: count, dtype: int64


In [13]:
# ============================================================
# SLICE MATRICES
# ============================================================

X_train_protein = X_protein_shared[train_idx]
X_val_protein = X_protein_shared[val_idx]
X_test_protein = X_protein_shared[test_idx]

X_train_genomic = X_genomic_shared.iloc[train_idx].reset_index(drop=True)
X_val_genomic = X_genomic_shared.iloc[val_idx].reset_index(drop=True)
X_test_genomic = X_genomic_shared.iloc[test_idx].reset_index(drop=True)

X_train_combined = X_combined_shared[train_idx]
X_val_combined = X_combined_shared[val_idx]
X_test_combined = X_combined_shared[test_idx]

y_train = y_shared[train_idx]
y_val = y_shared[val_idx]
y_test = y_shared[test_idx]

train_meta = shared_meta.iloc[train_idx].reset_index(drop=True)
val_meta = shared_meta.iloc[val_idx].reset_index(drop=True)
test_meta = shared_meta.iloc[test_idx].reset_index(drop=True)

print("Protein train/val/test:", X_train_protein.shape, X_val_protein.shape, X_test_protein.shape)
print("Genomic train/val/test:", X_train_genomic.shape, X_val_genomic.shape, X_test_genomic.shape)
print("Combined train/val/test:", X_train_combined.shape, X_val_combined.shape, X_test_combined.shape)

Protein train/val/test: (1264, 1024) (271, 1024) (271, 1024)
Genomic train/val/test: (1264, 356) (271, 356) (271, 356)
Combined train/val/test: (1264, 1380) (271, 1380) (271, 1380)


In [14]:
# ============================================================
# LEAKAGE CHECKS
# ============================================================

train_genes = set(train_meta["gene_id"])
val_genes = set(val_meta["gene_id"])
test_genes = set(test_meta["gene_id"])

print("Train-Val overlap:", len(train_genes & val_genes))
print("Train-Test overlap:", len(train_genes & test_genes))
print("Val-Test overlap:", len(val_genes & test_genes))

assert len(train_genes & val_genes) == 0
assert len(train_genes & test_genes) == 0
assert len(val_genes & test_genes) == 0

print("✅ No gene leakage in shared multimodal split.")

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0
✅ No gene leakage in shared multimodal split.


In [15]:
# ============================================================
# SAVE SHARED MULTIMODAL DATASET
# ============================================================

# Save metadata
train_meta.to_csv(PHASE3_DATA_DIR / "train_multimodal_metadata_v1.csv", index=False)
val_meta.to_csv(PHASE3_DATA_DIR / "val_multimodal_metadata_v1.csv", index=False)
test_meta.to_csv(PHASE3_DATA_DIR / "test_multimodal_metadata_v1.csv", index=False)

shared_meta.to_csv(PHASE3_DATA_DIR / "all_shared_multimodal_metadata_v1.csv", index=False)

# Save labels
np.save(PHASE3_DATA_DIR / "y_train_multimodal_v1.npy", y_train)
np.save(PHASE3_DATA_DIR / "y_val_multimodal_v1.npy", y_val)
np.save(PHASE3_DATA_DIR / "y_test_multimodal_v1.npy", y_test)

# Save protein matrices
np.save(PHASE3_FEATURE_DIR / "X_train_protein_protbert_sw_v1.npy", X_train_protein)
np.save(PHASE3_FEATURE_DIR / "X_val_protein_protbert_sw_v1.npy", X_val_protein)
np.save(PHASE3_FEATURE_DIR / "X_test_protein_protbert_sw_v1.npy", X_test_protein)

# Save genomic matrices
X_train_genomic.to_csv(PHASE3_FEATURE_DIR / "X_train_genomic_k3k4basic_v1.csv", index=False)
X_val_genomic.to_csv(PHASE3_FEATURE_DIR / "X_val_genomic_k3k4basic_v1.csv", index=False)
X_test_genomic.to_csv(PHASE3_FEATURE_DIR / "X_test_genomic_k3k4basic_v1.csv", index=False)

# Save combined matrices
np.save(PHASE3_FEATURE_DIR / "X_train_combined_protein_genomic_v1.npy", X_train_combined)
np.save(PHASE3_FEATURE_DIR / "X_val_combined_protein_genomic_v1.npy", X_val_combined)
np.save(PHASE3_FEATURE_DIR / "X_test_combined_protein_genomic_v1.npy", X_test_combined)

print("✅ Saved Phase 3.0 shared multimodal dataset.")

✅ Saved Phase 3.0 shared multimodal dataset.


In [16]:
# ============================================================
# SAVE FEATURE NAMES
# ============================================================

protein_feature_names = [
    f"protbert_sw_{i}"
    for i in range(X_protein_shared.shape[1])
]

genomic_feature_names = X_genomic_shared.columns.tolist()

combined_feature_names = protein_feature_names + genomic_feature_names

feature_name_info = {
    "n_protein_features": len(protein_feature_names),
    "n_genomic_features": len(genomic_feature_names),
    "n_combined_features": len(combined_feature_names),
    "protein_feature_names": protein_feature_names,
    "genomic_feature_names": genomic_feature_names,
    "combined_feature_names": combined_feature_names
}

with open(PHASE3_FEATURE_DIR / "phase3_0_feature_names_v1.json", "w") as f:
    json.dump(feature_name_info, f, indent=4)

print("Protein features:", len(protein_feature_names))
print("Genomic features:", len(genomic_feature_names))
print("Combined features:", len(combined_feature_names))

Protein features: 1024
Genomic features: 356
Combined features: 1380


In [17]:
# ============================================================
# SPLIT SUMMARY
# ============================================================

split_summary_records = []

for split_name, meta_df, y_arr, Xp, Xg, Xc in [
    ("train", train_meta, y_train, X_train_protein, X_train_genomic, X_train_combined),
    ("validation", val_meta, y_val, X_val_protein, X_val_genomic, X_val_combined),
    ("test", test_meta, y_test, X_test_protein, X_test_genomic, X_test_combined)
]:
    split_summary_records.append({
        "split": split_name,
        "n": len(meta_df),
        "n_negative": int((y_arr == 0).sum()),
        "n_positive": int((y_arr == 1).sum()),
        "positive_ratio": float((y_arr == 1).mean()),
        "n_protein_features": Xp.shape[1],
        "n_genomic_features": Xg.shape[1],
        "n_combined_features": Xc.shape[1],
        "unique_gene_id": int(meta_df["gene_id"].nunique())
    })

phase3_split_summary_df = pd.DataFrame(split_summary_records)

display(phase3_split_summary_df)

phase3_split_summary_df.to_csv(
    PHASE3_RESULT_DIR / "phase3_0_shared_multimodal_split_summary_v1.csv",
    index=False
)

,split,n,n_negative,n_positive,positive_ratio,n_protein_features,n_genomic_features,n_combined_features,unique_gene_id
0,train,1264,632,632,0.500000,1024,356,1380,1264
1,validation,271,135,136,0.501845,1024,356,1380,271
2,test,271,136,135,0.498155,1024,356,1380,271


In [18]:
# ============================================================
# PHASE 3.0 QC REPORT
# ============================================================

phase3_0_report = f"""
# Phase 3.0 — Shared Multimodal Dataset Preparation

## Objective

The purpose of Phase 3.0 was to create a shared multimodal dataset for protein + genomic integration.

The two modalities are:

1. Protein sequence representation:
   ProtBERT sliding-window embeddings

2. Genomic regulatory representation:
   K3 + K4 + Basic TSS-proximal handcrafted genomic features

## Input Sources

Protein source:
{PROTBERT_SW_EMBED_DIR}

Genomic source:
{GENOMIC_FEATURE_DIR}

## Shared Gene Set

Protein genes available: {protein_meta_std['gene_id'].nunique()}
Genomic genes available: {genomic_meta_std['gene_id'].nunique()}
Shared multimodal genes: {shared_meta.shape[0]}

Label distribution in shared set:
{shared_meta['label'].value_counts().sort_index().to_string()}

## Feature Dimensions

Protein features: {X_protein_shared.shape[1]}
Genomic features: {X_genomic_shared.shape[1]}
Combined features: {X_combined_shared.shape[1]}

## Shared Split Strategy

The shared multimodal dataset was split using:

- 70% train
- 15% validation
- 15% test
- stratified by label
- random_state = 42

## Split Summary

{phase3_split_summary_df.to_string(index=False)}

## Leakage Check

No gene_id overlap was found across train, validation, and test splits.

## Output Files

Metadata:
- train_multimodal_metadata_v1.csv
- val_multimodal_metadata_v1.csv
- test_multimodal_metadata_v1.csv

Protein features:
- X_train_protein_protbert_sw_v1.npy
- X_val_protein_protbert_sw_v1.npy
- X_test_protein_protbert_sw_v1.npy

Genomic features:
- X_train_genomic_k3k4basic_v1.csv
- X_val_genomic_k3k4basic_v1.csv
- X_test_genomic_k3k4basic_v1.csv

Combined features:
- X_train_combined_protein_genomic_v1.npy
- X_val_combined_protein_genomic_v1.npy
- X_test_combined_protein_genomic_v1.npy

## Next Step

The next step is Phase 3.1: train and evaluate multimodal integration models.

Recommended models:
- Logistic Regression
- SVM RBF
- Random Forest
- LightGBM
- Soft Voting
- Stacking

Recommended comparison:
- protein-only on shared split
- genomic-only on shared split
- combined protein + genomic on shared split
"""

report_path = PHASE3_REPORT_DIR / "phase3_0_shared_multimodal_dataset_report.md"

with open(report_path, "w") as f:
    f.write(phase3_0_report)

print(phase3_0_report)
print("Saved:", report_path)


# Phase 3.0 — Shared Multimodal Dataset Preparation

## Objective

The purpose of Phase 3.0 was to create a shared multimodal dataset for protein + genomic integration.

The two modalities are:

1. Protein sequence representation:
   ProtBERT sliding-window embeddings

2. Genomic regulatory representation:
   K3 + K4 + Basic TSS-proximal handcrafted genomic features

## Input Sources

Protein source:
/content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/embeddings

Genomic source:
/content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/features

## Shared Gene Set

Protein genes available: 1820
Genomic genes available: 1806
Shared multimodal genes: 1806

Label distribution in shared set:
label
0    903
1    903

## Feature Dimensions

Protein features: 1024
Genomic features: 356
Combined features: 1380

## Shared Split Strategy

The shared multimodal dataset was split using:

- 70% train
- 15% validation
- 15% test
- stratified

In [19]:
print("Shared genes:", shared_meta.shape)
print(shared_meta["label"].value_counts())

display(phase3_split_summary_df)

print("X_train_protein:", X_train_protein.shape)
print("X_train_genomic:", X_train_genomic.shape)
print("X_train_combined:", X_train_combined.shape)

Shared genes: (1806, 12)
label
0    903
1    903
Name: count, dtype: int64


,split,n,n_negative,n_positive,positive_ratio,n_protein_features,n_genomic_features,n_combined_features,unique_gene_id
0,train,1264,632,632,0.500000,1024,356,1380,1264
1,validation,271,135,136,0.501845,1024,356,1380,271
2,test,271,136,135,0.498155,1024,356,1380,271


X_train_protein: (1264, 1024)
X_train_genomic: (1264, 356)
X_train_combined: (1264, 1380)
